# Confusion-matrix metrics for `prediction-cm/`

Crawls every confusion-matrix CSV under `prediction-cm/` and prints every usual classification metric
calculated from it, together with the numbers that go into each calculation.

Both CSV formats in the folder are handled:
- **old runs** (`old*`): `true_class, <predicted classes...>`
- **new runs**: `<metadata columns...>, true_class, <predicted classes...>`

Rows are true classes, columns are predicted classes.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("prediction-cm")       # folder to crawl, relative to this notebook (repo root)

# Filters: None = everything. Run and model match exactly; scenario matches a substring,
# so "gaussian-30" picks up "gaussian-30", "gaussian-30-all" and "gaussian-30-noisy".
RUN_FILTER      = None             # e.g. ["original_with-smote_with-pca"]
MODEL_FILTER    = None             # e.g. ["rf", "xgb"]
SCENARIO_FILTER = None             # e.g. ["clean", "gaussian-30"]

SHOW_CM      = True                # also print the raw confusion matrix
BENIGN_LABEL = "Benign"            # used for the binary Benign-vs-Attack view
SAVE_SUMMARY = False               # write the summary table to prediction-cm-summary.csv

assert ROOT.is_dir(), f"{ROOT.resolve()} not found: open this notebook from the repo root"


## Metric definitions

For class *k* (one-vs-rest), with confusion matrix *C*, *N* total samples,
*t_k* = row sum (true count) and *p_k* = column sum (predicted count):

| Metric | Formula |
|---|---|
| TP_k | C[k, k] |
| FP_k | p_k − TP_k (predicted *k*, actually another class) |
| FN_k | t_k − TP_k (actually *k*, predicted as another class) |
| TN_k | N − TP_k − FP_k − FN_k |
| Precision (PPV) | TP / (TP + FP) |
| Recall (sensitivity, TPR) | TP / (TP + FN) |
| F1 | 2·P·R / (P + R) = 2TP / (2TP + FP + FN) |
| Specificity (TNR) | TN / (TN + FP) |
| FPR | FP / (FP + TN) = 1 − specificity |
| FNR (miss rate) | FN / (FN + TP) = 1 − recall |
| NPV | TN / (TN + FN) |
| Jaccard (IoU) | TP / (TP + FP + FN) |
| **Accuracy** | Σ TP / N |
| **Balanced accuracy** | mean recall over classes with support > 0 |
| **Macro avg** | plain mean over classes (each class counts equally) |
| **Weighted avg** | Σ support_k · metric_k / N |
| **Micro avg** | pool TP/FP/FN over all classes; equals accuracy for single-label multiclass |
| **Cohen's κ** | (p_o − p_e) / (1 − p_e), with p_o = accuracy, p_e = Σ t_k·p_k / N² |
| **MCC** | (c·N − Σ t_k·p_k) / √((N² − Σ p_k²)(N² − Σ t_k²)), c = Σ TP |
| **Detection rate** | Benign vs Attack: attacks flagged as *any* attack / all attacks |
| **False alarm rate** | Benign vs Attack: benign flagged as an attack / all benign |

These conventions match scikit-learn:
- A division by zero gives 0 (sklearn's `zero_division=0`).
- Macro F1 is the mean of the per-class F1 scores, not the F1 of macro precision and macro recall.
- A class with no true samples and no predictions is left out of the averages.
- A class with no true samples but some predictions stays in, with precision, recall and F1 all 0.

The last cell checks these numbers against sklearn.

In [2]:
def safe_div(a, b):
    """a / b, but 0 where b == 0 (sklearn's zero_division=0)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    out = np.divide(a, b, out=np.zeros(np.broadcast(a, b).shape), where=b != 0)
    return out if out.ndim else float(out)


def load_cm(path):
    """Read one CSV. Returns (info, cm DataFrame), or None if it isn't a confusion matrix."""
    df = pd.read_csv(path)
    if "true_class" not in df.columns:
        return None
    split = df.columns.get_loc("true_class")
    meta = df.iloc[0, :split].to_dict()                 # metadata is repeated on every row
    cm = df.iloc[:, split:].set_index("true_class")
    labels = list(cm.index) + [c for c in cm.columns if c not in cm.index]
    cm = cm.reindex(index=labels, columns=labels, fill_value=0).astype("int64")
    rel = path.relative_to(ROOT)
    info = {
        "run": rel.parts[0],
        "model": rel.parent.name,
        "scenario": path.stem.removesuffix("-confusion-matrix"),
        "path": str(rel),
        "meta": {k: v for k, v in meta.items() if k not in ("experiment", "model", "scenario")},
    }
    return info, cm


def selected(info):
    return ((RUN_FILTER is None or info["run"] in RUN_FILTER)
            and (MODEL_FILTER is None or info["model"] in MODEL_FILTER)
            and (SCENARIO_FILTER is None or any(s in info["scenario"] for s in SCENARIO_FILTER)))


def iter_cms():
    """Yield (info, cm) for every confusion-matrix CSV under ROOT that passes the filters."""
    for path in sorted(ROOT.rglob("*.csv")):            # '*.csv' skips the ':Zone.Identifier' files
        loaded = load_cm(path)
        if loaded and selected(loaded[0]):
            yield loaded


In [3]:
def evaluate(cm, benign=BENIGN_LABEL):
    """Every metric for one confusion matrix.
    Returns (per-class DataFrame, list of (section, name, value, calculation) rows)."""
    C = cm.to_numpy()
    N = int(C.sum())
    t = C.sum(axis=1)                       # true count per class      (row sums)
    p = C.sum(axis=0)                       # predicted count per class (column sums)
    tp = np.diag(C)
    fp, fn = p - tp, t - tp
    tn = N - tp - fp - fn

    pc = pd.DataFrame({
        "support": t, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision":   safe_div(tp, tp + fp),
        "recall":      safe_div(tp, tp + fn),
        "f1":          safe_div(2 * tp, 2 * tp + fp + fn),
        "specificity": safe_div(tn, tn + fp),
        "FPR":         safe_div(fp, fp + tn),
        "FNR":         safe_div(fn, fn + tp),
        "NPV":         safe_div(tn, tn + fn),
        "jaccard":     safe_div(tp, tp + fp + fn),
    }, index=cm.index)
    pc = pc[(pc.support > 0) | (pc.FP > 0)]   # drop classes with no samples and no predictions
    k, correct = len(pc), int(tp.sum())

    rows = []
    def add(section, name, value, calc):
        rows.append((section, name, float(value), calc))

    # Overall
    acc = correct / N
    add("overall", "Accuracy", acc, f"Σ TP / N = {correct:,} / {N:,}")
    add("overall", "Error rate", 1 - acc, f"(N − Σ TP) / N = {N - correct:,} / {N:,}")
    has = pc[pc.support > 0]
    add("overall", "Balanced accuracy", has.recall.mean(),
        f"mean recall over {len(has)} classes with support > 0 = {has.recall.sum():.4f} / {len(has)}")

    # Macro / weighted
    for m in ["precision", "recall", "f1"]:
        add("macro", f"Macro {m}", pc[m].mean(), f"mean of {k} per-class {m} = {pc[m].sum():.4f} / {k}")
    for m in ["precision", "recall", "f1"]:
        s = (pc.support * pc[m]).sum()
        note = "  (always = accuracy)" if m == "recall" else ""
        add("weighted", f"Weighted {m}", s / N, f"Σ support·{m} / N = {s:,.2f} / {N:,}{note}")

    # Micro
    TP, FP, FN = int(pc.TP.sum()), int(pc.FP.sum()), int(pc.FN.sum())
    add("micro", "Micro precision", safe_div(TP, TP + FP), f"Σ TP / Σ(TP + FP) = {TP:,} / {TP + FP:,}")
    add("micro", "Micro recall", safe_div(TP, TP + FN), f"Σ TP / Σ(TP + FN) = {TP:,} / {TP + FN:,}")
    add("micro", "Micro f1", safe_div(2 * TP, 2 * TP + FP + FN),
        f"2·Σ TP / (2·Σ TP + Σ FP + Σ FN) = {2 * TP:,} / {2 * TP + FP + FN:,}")

    # Agreement: Cohen's kappa and MCC
    tp_sum = float((t * p).sum())
    pe = tp_sum / N**2
    add("agreement", "Cohen's kappa", safe_div(acc - pe, 1 - pe),
        f"(p_o − p_e) / (1 − p_e) = ({acc:.4f} − {pe:.4f}) / (1 − {pe:.4f})")
    num = correct * N - tp_sum
    den = np.sqrt((N**2 - float((p**2).sum())) * (N**2 - float((t**2).sum())))
    add("agreement", "MCC", safe_div(num, den),
        f"(c·N − Σ t·p) / √((N² − Σ p²)(N² − Σ t²)) = {num:.4e} / {den:.4e}")

    # Binary view: Benign vs any attack (standard IDS numbers)
    if benign in cm.index:
        b = cm.index.get_loc(benign)
        TNb = int(C[b, b])                  # benign  -> benign
        FPb = int(t[b] - C[b, b])           # benign  -> some attack   (false alarm)
        FNb = int(p[b] - C[b, b])           # attack  -> benign        (missed attack)
        TPb = N - TNb - FPb - FNb           # attack  -> any attack class
        add("binary", "Detection rate (attack recall)", safe_div(TPb, TPb + FNb),
            f"TP / (TP + FN) = {TPb:,} / {TPb + FNb:,}")
        add("binary", "False alarm rate (benign FPR)", safe_div(FPb, FPb + TNb),
            f"FP / (FP + TN) = {FPb:,} / {FPb + TNb:,}")
        add("binary", "Missed attack rate (FNR)", safe_div(FNb, TPb + FNb),
            f"FN / (TP + FN) = {FNb:,} / {TPb + FNb:,}")
        add("binary", "Attack precision", safe_div(TPb, TPb + FPb),
            f"TP / (TP + FP) = {TPb:,} / {TPb + FPb:,}")
        add("binary", "Attack f1", safe_div(2 * TPb, 2 * TPb + FPb + FNb),
            f"2TP / (2TP + FP + FN) = {2 * TPb:,} / {2 * TPb + FPb + FNb:,}")
        add("binary", "Binary accuracy", (TPb + TNb) / N,
            f"(TP + TN) / N = {TPb + TNb:,} / {N:,}")
    return pc, rows


In [4]:
SECTIONS = {
    "overall":   "Overall",
    "macro":     "Macro average (every class counts equally)",
    "weighted":  "Weighted average (each class weighted by its support)",
    "micro":     "Micro average (TP/FP/FN pooled over classes)",
    "agreement": "Agreement",
    "binary":    f"Binary view: {BENIGN_LABEL} vs Attack (all attack classes merged)",
}

def print_report(info, cm, pc, rows):
    line = "=" * 110
    print(line)
    print(f"RUN: {info['run']}   MODEL: {info['model']}   SCENARIO: {info['scenario']}")
    print(f"FILE: {ROOT / info['path']}")
    if info["meta"]:
        print("META: " + ", ".join(f"{k}={v}" for k, v in info["meta"].items()))
    print(line)
    print(f"N = {int(cm.to_numpy().sum()):,} samples, {len(pc)} classes")

    if SHOW_CM:
        print("\nConfusion matrix (rows = true, columns = predicted; column [i] = row [i]):")
        shown = cm.copy()
        shown.index = [f"[{i}] {c}" for i, c in enumerate(cm.index)]
        shown.columns = [f"[{i}]" for i in range(len(cm.columns))]
        print(shown.to_string(formatters={c: "{:,}".format for c in shown.columns}))

    print("\nPer-class metrics (one-vs-rest):")
    ints = ["support", "TP", "FP", "FN", "TN"]
    print(pc.rename_axis(None).to_string(formatters={c: "{:,}".format for c in ints}, float_format="{:.4f}".format))

    for section, title in SECTIONS.items():
        sec = [r for r in rows if r[0] == section]
        if sec:
            print(f"\n{title}:")
            for _, name, value, calc in sec:
                print(f"  {name:<32}{value:>8.4f}   = {calc}")
    print("\n")


## Crawl and print everything

In [5]:
summary_rows = []
for info, cm in iter_cms():
    pc, rows = evaluate(cm)
    print_report(info, cm, pc, rows)
    summary_rows.append({"run": info["run"], "model": info["model"], "scenario": info["scenario"],
                         "N": int(cm.to_numpy().sum()), "classes": len(pc),
                         **{name: value for _, name, value, _ in rows}})

summary = pd.DataFrame(summary_rows)
print(f"Evaluated {len(summary)} confusion matrices")


RUN: minimized_with-smote_with-pca   MODEL: knn   SCENARIO: clean
FILE: prediction-cm/minimized_with-smote_with-pca/knn/clean.csv
META: classes=minimized, smote=with-smote, pca=with-pca, classifier=KNeighborsClassifier, pca_components=26, pca_variance_percentage=95.73429870605467, smote_k_neighbors=3, param_knn__n_neighbors=1, param_knn__metric=manhattan, noise_type=clean, intensity=0.0
N = 2,303,097 samples, 6 classes

Confusion matrix (rows = true, columns = predicted; column [i] = row [i]):
                        [0]    [1]    [2]     [3]    [4] [5]
[0] Benign        2,003,690      5     41      23 28,917   8
[1] Bot                   4 28,903      0       0      0   0
[2] Brute Force          34      0 18,895      11      1  34
[3] DoS                  11      0      5 194,372      4   0
[4] Infiltration     24,718      0      1       0  3,401   2
[5] SQL Injection         4      0      1       0      0  12

Per-class metrics (one-vs-rest):
                support        TP     FP

## Summary: one row per confusion matrix

In [6]:
with pd.option_context("display.max_rows", None, "display.max_columns", None,
                       "display.float_format", "{:.4f}".format):
    display(summary)

if SAVE_SUMMARY:
    out = Path("prediction-cm-summary.csv")
    summary.to_csv(out, index=False)
    print(f"saved {out}")


,run,model,scenario,N,classes,Accuracy,Error rate,Balanced accuracy,Macro precision,Macro recall,Macro f1,Weighted precision,Weighted recall,Weighted f1,Micro precision,Micro recall,Micro f1,Cohen's kappa,MCC,Detection rate (attack recall),False alarm rate (benign FPR),Missed attack rate (FNR),Attack precision,Attack f1,Binary accuracy
0,minimized_with-smote_with-pca,knn,clean,2303097,6,0.9766,0.0234,0.8013,0.7174,0.8013,0.7374,0.9783,0.9766,0.9774,0.9766,0.9766,0.9766,0.8914,0.8914,0.9084,0.0143,0.0916,0.8944,0.9014,0.9767
1,minimized_with-smote_with-pca,knn,gaussian-10,2303097,6,0.9652,0.0348,0.7539,0.6676,0.7539,0.6566,0.9675,0.9652,0.9662,0.9652,0.9652,0.9652,0.8345,0.8347,0.8364,0.0170,0.1636,0.8676,0.8517,0.9658
2,minimized_with-smote_with-pca,knn,gaussian-20,2303097,6,0.9534,0.0466,0.6863,0.6452,0.6863,0.6165,0.9562,0.9534,0.9543,0.9534,0.9534,0.9534,0.7732,0.7743,0.7621,0.0198,0.2379,0.8365,0.7976,0.9546
3,minimized_with-smote_with-pca,knn,gaussian-30,2303097,6,0.9417,0.0583,0.6378,0.6227,0.6378,0.5776,0.9445,0.9417,0.9419,0.9417,0.9417,0.9417,0.7086,0.7113,0.6874,0.0226,0.3126,0.8019,0.7403,0.9434
4,minimized_with-smote_with-pca,knn,missing-10,2303097,6,0.9647,0.0353,0.7315,0.6767,0.7315,0.6574,0.9679,0.9647,0.9660,0.9647,0.9647,0.9647,0.8305,0.8310,0.8233,0.0162,0.1767,0.8710,0.8465,0.9649
5,minimized_with-smote_with-pca,knn,missing-20,2303097,6,0.9529,0.0471,0.6806,0.6679,0.6806,0.6221,0.9577,0.9529,0.9542,0.9529,0.9529,0.9529,0.7664,0.7687,0.7385,0.0180,0.2615,0.8452,0.7883,0.9534
6,minimized_with-smote_with-pca,knn,missing-30,2303097,6,0.9412,0.0588,0.6223,0.6604,0.6223,0.5871,0.9478,0.9412,0.9420,0.9412,0.9412,0.9412,0.6986,0.7039,0.6549,0.0198,0.3451,0.8145,0.7260,0.9420
7,minimized_with-smote_with-pca,knn,multiplicative-10,2303097,6,0.9682,0.0318,0.7620,0.6761,0.7620,0.6627,0.9709,0.9682,0.9694,0.9682,0.9682,0.9682,0.8491,0.8493,0.8527,0.0158,0.1473,0.8780,0.8652,0.9688
8,minimized_with-smote_with-pca,knn,multiplicative-20,2303097,6,0.9598,0.0402,0.7034,0.6674,0.7034,0.6351,0.9636,0.9598,0.9613,0.9598,0.9598,0.9598,0.8056,0.8064,0.7968,0.0172,0.2032,0.8606,0.8275,0.9610
9,minimized_with-smote_with-pca,knn,multiplicative-30,2303097,6,0.9512,0.0488,0.6532,0.6570,0.6532,0.6059,0.9560,0.9512,0.9526,0.9512,0.9512,0.9512,0.7591,0.7611,0.7405,0.0187,0.2595,0.8406,0.7874,0.9530


## Quick comparison: one metric as scenario × model, per run

In [7]:
METRIC = "Macro f1"    # any summary column, e.g. "Accuracy", "Weighted f1", "MCC", "Detection rate (attack recall)"

for run, g in summary.groupby("run"):
    print(f"\n{METRIC}: {run}")
    print(g.pivot(index="scenario", columns="model", values=METRIC).to_string(float_format="{:.4f}".format))



Macro f1: minimized_with-smote_with-pca
model                knn  logreg     rf    svm    xgb
scenario                                             
clean             0.7374  0.6506 0.7285 0.6788 0.7209
gaussian-10       0.6566  0.5082 0.7042 0.6515 0.6518
gaussian-20       0.6165  0.4254 0.6695 0.6186 0.6081
gaussian-30       0.5776  0.3643 0.6317 0.5875 0.5642
missing-10        0.6574  0.6015 0.6721 0.6397 0.6703
missing-20        0.6221  0.5554 0.6347 0.6075 0.6314
missing-30        0.5871  0.5137 0.5968 0.5747 0.5944
multiplicative-10 0.6627  0.5759 0.6903 0.6423 0.6592
multiplicative-20 0.6351  0.5217 0.6474 0.6161 0.6259
multiplicative-30 0.6059  0.4786 0.6099 0.5886 0.5902
uniform-10        0.6561  0.5077 0.7013 0.6503 0.6497
uniform-20        0.6161  0.4250 0.6635 0.6190 0.6067
uniform-30        0.5763  0.3639 0.6392 0.5886 0.5633

Macro f1: old-minimized_with-smote_with-pca
model                      knn  logreg     rf    svm    xgb
scenario                                    

## Sanity check against scikit-learn

Each non-empty CM cell becomes one sample weighted by its count, so sklearn sees exactly the same predictions.
Checks the per-class precision/recall/F1 and all overall, macro, weighted and micro metrics, plus κ and MCC.

In [8]:
import warnings
from sklearn import metrics as skm

def sklearn_metrics(cm):
    C = cm.to_numpy()
    yt, yp = np.nonzero(C)                  # one weighted "sample" per non-empty cell
    w = C[yt, yp]
    out = {
        "Accuracy":          skm.accuracy_score(yt, yp, sample_weight=w),
        "Balanced accuracy": skm.balanced_accuracy_score(yt, yp, sample_weight=w),
        "Cohen's kappa":     skm.cohen_kappa_score(yt, yp, sample_weight=w),
        "MCC":               skm.matthews_corrcoef(yt, yp, sample_weight=w),
    }
    for avg in ["macro", "weighted", "micro"]:
        prf = skm.precision_recall_fscore_support(yt, yp, average=avg, sample_weight=w, zero_division=0)
        for name, v in zip(["precision", "recall", "f1"], prf[:3]):
            out[f"{avg.capitalize()} {name}"] = v
    per_class = skm.precision_recall_fscore_support(yt, yp, average=None, sample_weight=w, zero_division=0)
    return out, np.vstack(per_class[:3]).T

checked, mismatches = 0, 0
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for info, cm in iter_cms():
        pc, rows = evaluate(cm)
        ours = {name: v for _, name, v, _ in rows}
        theirs, per_class = sklearn_metrics(cm)
        for name, v in theirs.items():
            if not np.isclose(ours[name], v, rtol=0, atol=1e-9):
                mismatches += 1
                print(f"MISMATCH {info['path']} | {name}: ours={ours[name]:.6f} sklearn={v:.6f}")
        if not np.allclose(pc[["precision", "recall", "f1"]].to_numpy(), per_class, rtol=0, atol=1e-9):
            mismatches += 1
            print(f"MISMATCH {info['path']} | per-class precision/recall/f1")
        checked += 1

print(f"Checked {checked} files: " + ("everything matches sklearn ✓" if mismatches == 0 else f"{mismatches} mismatches"))


Checked 505 files: everything matches sklearn ✓
